# 🧠 Notebook 02: Floating Point and Soft Math

## 1. Purpose + Scope

This notebook demonstrates T81's deterministic floating-point arithmetic:

*   **T81Float Layout**: Mantissa and exponent representation.
*   **NaE State**: Not-an-Entity handling (NaN equivalent).
*   **Strict vs Host-Dependent Operations**: Ensuring determinism.
*   **dmath Backend**: Software-defined math library.

## 2. Spec References

*   `spec/t81-data-types.md`
*   `include/t81/core/T81Float.hpp`
*   `include/t81/core/detail/dmath.hpp`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: T81Float operations are software-defined to ensure bit-exact reproducibility across architectures, avoiding hardware FPU inconsistencies.

## 4. Reproducibility Setup

Ensure `t81_python` is built and available in `PYTHONPATH`.

In [ ]:
import sys
import os

build_dir = os.path.abspath(os.path.join(os.getcwd(), "../build"))
if build_dir not in sys.path:
    sys.path.append(build_dir)

try:
    from t81_python import Float
    print("✅ t81_python Float loaded.")
except ImportError:
    print("❌ Failed to load t81_python.")
    sys.exit(1)

## 5. Exploratory Code: T81Float

`Float` wraps `T81Float27_9` (or similar depending on binding). It provides deterministic floating-point math.

In [ ]:
# Basic arithmetic
x = Float(3.14159)
y = Float(2.71828)

sum_val = x + y
prod_val = x * y

print(f"Sum: {sum_val}")
print(f"Product: {prod_val}")

# Note: The Python binding likely converts to double for initialization and repr,
# but internal operations use the T81Float logic.

## 6. Determinism Verification

Since standard floating point can vary, T81Float uses integer-based logic internally. We verify that operations are consistent.

In [ ]:
def check_determinism(val1, val2):
    res1 = val1 / val2
    res2 = val1 / val2
    # Repr should be identical
    assert str(res1) == str(res2)
    print(f"Division Result: {res1}")

check_determinism(x, y)

## 7. dmath Backend Commentary

The `dmath` backend implements transcendental functions (sin, cos, exp) using fixed-point algorithms or Taylor series expansions with integer arithmetic to guarantee that `sin(x)` returns the exact same bit pattern on x86, ARM, and RISC-V.

## 8. Failure Mode Demonstration

Division by zero in floating point often results in Infinity or NaN. T81 defines specific behaviors for these cases (NaE).

In [ ]:
try:
    z = Float(0.0)
    res = x / z
    print(f"Result of division by zero: {res}")
except Exception as e:
    print(f"Caught exception: {e}")

## 9. Architectural Commentary

Strict determinism comes at a performance cost compared to hardware FPU instructions. T81 accepts this trade-off to ensure consensus-critical code behaves identically everywhere. JIT compilers for T81 can optimize this by proving when safe hardware instructions can be used (e.g., when exact precision is not required by policy).